# 🔵 Clustering - Hands-On Notebook
**QLSC 612 · Machine Learning Module · Part II**

In this notebook we will:
1. Load fMRI connectivity data during a motor task scan
2. Visualise it using PCA
3. Apply K-Means clustering
4. Evaluate clusters with **internal** (silhouette) and **external** (ARI) metrics
5. Explore what happens when we change k

---
> **⏱ Estimated time:** ~45 minutes  
> **💡 Tip:** Run each cell with `Shift+Enter`. Read the comments — they explain *why*, not just *how*.

## 0. Imports & Setup

Run this cell first. It loads all the packages we will need.

In [ ]:
# --- Standard libraries ---
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- scikit-learn ---
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
)

# --- Nice plots ---
sns.set_theme(style='whitegrid', font_scale=1.1)
PALETTE = sns.color_palette('tab10')

print('✅  All imports successful!')

---
## 1. Load the Data

We load the already extracted connectivity data.

- **X** : functional connectivity features (correlation between brain regions) for each time point  
- **y** : the task label: subject at rest, or subejct in task

We are asking: *can K-means automatically recover which cognitive state a time point belongs to — without ever seeing the task labels?*

In [ ]:
DATA = np.load(f"./fmri_task-motor_data.npy", allow_pickle=True).item()

X_raw = DATA["X"] # shape: (n_samples, n_features) = (n_time, n_connections)
y = DATA["y"] # shape: (n_samples,) = (n_time,) with values 0 (rest) or 1 (task)
unique_labels = ["rest", "task"]

print(f"\nData shape:  X = {X_raw.shape}  (n_time × n_connections)")
print(f"Target shape:  y = {y.shape}  (n_subjects,)")
print(f"Cognitive States: {unique_labels}")

---
## 2. Preprocessing: Scale + PCA

Before clustering we must:
1. **Standardise** (z-score) features — K-means uses distances, so scale matters
2. **Reduce with PCA** — connectivity data has thousands of features; distances become meaningless in such high dimensions (curse of dimensionality!)

### 🤔 Think about it
> *What might happen if we ran K-means on the raw 4,950-dimensional features?*

In [ ]:
# ── Step 1: scale ─────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# ── Step 2: PCA ───────────────────────────────────────────────────────────────
N_COMPONENTS = 50   # ← try changing this and see what happens!
pca = PCA(n_components=N_COMPONENTS, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# How much variance do we keep?
cumvar = np.cumsum(pca.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: explained variance curve
axes[0].plot(range(1, N_COMPONENTS+1), pca.explained_variance_ratio_, 'o-', color='teal', ms=4)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance explained')
axes[0].set_title('Scree plot')

# Right: cumulative
axes[1].plot(range(1, N_COMPONENTS+1), cumvar, 'o-', color='coral', ms=4)
axes[1].axhline(0.80, ls='--', color='grey', lw=1, label='80% threshold')
axes[1].set_xlabel('Number of PCs')
axes[1].set_ylabel('Cumulative variance explained')
axes[1].set_title('Cumulative variance')
axes[1].legend()

plt.suptitle(f'PCA on connectivity data ({N_COMPONENTS} components)', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Variance kept with {N_COMPONENTS} PCs: {cumvar[-1]*100:.1f}%')
print(f'Dimensionality: {X_raw.shape[1]} → {N_COMPONENTS}')

### Visualise: Can we already see clusters in 2D PCA space?

Let's color each point by its **true rest/task label** and look at the first two PCs.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for i, state in enumerate(unique_labels):
    mask = (y == i)
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               label=state, alpha=0.7, s=25, color=PALETTE[i % 10])

ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.set_title('PCA (2D) - colored by true states')
ax.legend(bbox_to_anchor=(1.01, 1), title='State', fontsize=9)
plt.tight_layout()
plt.show()

print('💬  Discussion: Can you see separation between rest and task?')
print('    What might be driving this pattern?')

---
## 3. K-Means Clustering

Now let's run K-means. We set `k` equal to the number of states — the question is whether the algorithm can recover those groups.

In [ ]:
K = len(unique_labels)  # number of clusters = number of states
print(f'Running K-means with k = {K} ...')

km = KMeans(n_clusters=K, n_init=20, random_state=42, max_iter=500)
cluster_labels = km.fit_predict(X_pca)   # ← we cluster on the PCA-reduced data

print(f'✅  Done! Inertia (within-cluster SS): {km.inertia_:.1f}')
print(f'Cluster sizes: {np.bincount(cluster_labels)}')

In [ ]:
# ── Visualise cluster assignments ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: true states
for i, state in enumerate(unique_labels):
    mask = (y == i)
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    label=state, alpha=0.7, s=25, color=PALETTE[i % 10])
axes[0].set_title('True state labels')
axes[0].set_xlabel('PC 1'); axes[0].set_ylabel('PC 2')
axes[0].legend(fontsize=8, title='State', bbox_to_anchor=(1.01,1))

# Right: K-means clusters
for k in range(K):
    mask = (cluster_labels == k)
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    label=f'Cluster {k}', alpha=0.7, s=25, color=PALETTE[k % 10])
# plot centroids in PCA space
centroids_pca = km.cluster_centers_   # these are already in PCA space
axes[1].scatter(centroids_pca[:, 0], centroids_pca[:, 1],
                marker='*', s=80, c='black', zorder=5, label='Centroids')
axes[1].set_title(f'K-Means assignments (k={K})')
axes[1].set_xlabel('PC 1'); axes[1].set_ylabel('PC 2')
axes[1].legend(fontsize=8, bbox_to_anchor=(1.01,1))

plt.tight_layout()
plt.show()

---
## 4. Evaluation

### 4a. Internal Validation: Silhouette Score

This tells us about cluster *geometry*: are points closer to their own cluster than to others?  
Range: **−1** (bad) to **+1** (perfect).

> ⚠️ *High silhouette score ≠ meaningful biological clusters!*

In [ ]:
# Compute on a sample for speed (silhouette is O(n²))
N_SAMPLE = min(2000, len(X_pca))
idx = np.random.choice(len(X_pca), N_SAMPLE, replace=False)

si = silhouette_score(X_pca[idx], cluster_labels[idx])
print(f'Silhouette Score (k={K}): {si:.3f}')

if si > 0.75:
    print('→  Well-separated clusters!')
elif si > 0.25:
    print('→  Moderate cluster structure.')
else:
    print('→  Weak cluster structure — clusters overlap considerably.')

### 4b. External Validation: Adjusted Rand Index (ARI)

Now we compare our **cluster assignments** to the **true state labels**.  
- **ARI = 1.0** → perfect match  
- **ARI ≈ 0** → no better than random

This requires known labels — which we have (the rest/task labels). In a pure discovery scenario you might not have them.

In [ ]:
ari = adjusted_rand_score(y, cluster_labels)
print(f'Adjusted Rand Index (ARI):  {ari:.3f}')
print()
if ari > 0.75:
    print('🎉  Clusters closely match true cognitive states — strong state effect in the data!')
elif ari > 0.25:
    print('📊  Partial overlap — some rest and task samples separate well, others are mixed.')
else:
    print('🔍  Low ARI — clusters do not align with participants cognitive states.')
    print('    Possible reasons: Participants may not be following the expected cognitive states, or other factors drive variation.')

---
## 5. Choosing k - Elbow Plot + Silhouette

In real data you often don't know k. Let's explore a range and use metrics to guide our choice.

In [ ]:
k_range = range(2, min(15, len(unique_labels)+5))

inertias, si_scores, ari_scores = [], [], []

for k in k_range:
    km_k = KMeans(n_clusters=k, n_init=10, random_state=42)
    labs = km_k.fit_predict(X_pca)
    inertias.append(km_k.inertia_)
    si_scores.append(silhouette_score(X_pca[idx], labs[idx]))
    ari_scores.append(adjusted_rand_score(y, labs))
    print(f'  k={k:2d} | inertia={km_k.inertia_:8.0f} | SI={si_scores[-1]:.3f} | ARI={ari_scores[-1]:.3f}')

print('\n✅  Done!')

In [ ]:
k_list = list(k_range)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Elbow plot
axes[0].plot(k_list, inertias, 'o-', color='teal', lw=2)
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow plot')

# Silhouette
axes[1].plot(k_list, si_scores, 's-', color='coral', lw=2)
axes[1].axhline(0, ls='--', color='grey', lw=1)
best_k_si = k_list[np.argmax(si_scores)]
axes[1].axvline(best_k_si, ls=':', color='coral', lw=1.5, label=f'Best k={best_k_si}')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette score')
axes[1].set_title('Silhouette score (internal)')
axes[1].legend()

# ARI (external) — only meaningful if you have ground truth
axes[2].plot(k_list, ari_scores, 'D-', color='purple', lw=2)
best_k_ari = k_list[np.argmax(ari_scores)]
axes[2].axvline(best_k_ari, ls=':', color='purple', lw=1.5, label=f'Best k={best_k_ari}')
axes[2].axvline(len(unique_labels), ls='--', color='grey', lw=1.5, label=f'True k={len(unique_labels)}')
axes[2].set_xlabel('k'); axes[2].set_ylabel('ARI')
axes[2].set_title('ARI vs true cognitive states (external)')
axes[2].legend()

plt.suptitle('How does k affect clustering quality?', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f'\nBest k by Silhouette:  {best_k_si}')
print(f'Best k by ARI:         {best_k_ari}')
print(f'True number of states:  {len(unique_labels)}')
print('\n💬  Do these three numbers agree? What does that tell you?')

---
## 6. ✍️ Exercises

Try these on your own. There's no single correct answer — the goal is to build intuition.

### Exercise A: Change the number of PCA components
Go back to Section 2 and change `N_COMPONENTS` to 10, 20, or 100.  
How does this affect the ARI and silhouette score?

### Exercise B: Cluster without PCA
Run K-means directly on `X_scaled` (no PCA). Compare the silhouette and ARI to the PCA version.  
This might be slow — start with a small k!

```python
# Your code here
km_nopca = KMeans(n_clusters=K, n_init=10, random_state=42)
# ...
```

### Exercise C: Use the sklearn Pipeline
Wrap everything into a clean pipeline:

```python
pipe = make_pipeline(
    StandardScaler(),
    PCA(n_components=50, random_state=42),
    KMeans(n_clusters=K, n_init=20, random_state=42)
)
labels_pipe = pipe.fit_predict(X_raw)
print('ARI:', adjusted_rand_score(y, labels_pipe))
```

---
## 📋 Summary Cheatsheet

```python
# ── Full clustering pipeline in scikit-learn ─────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.pipeline import make_pipeline

# 1. Build and run the pipeline
pipe = make_pipeline(
    StandardScaler(),
    PCA(n_components=50, random_state=42),
    KMeans(n_clusters=2, n_init=20, random_state=42),
)
labels = pipe.fit_predict(X)

# 2. Get the PCA-transformed data for plotting/metrics
X_pca = pipe[:-1].transform(X)   # apply scaler + PCA, skip KMeans

# 3. Internal validation (no labels needed)
si = silhouette_score(X_pca, labels)          # higher = better

# 4. External validation (requires ground-truth labels y_true)
ari = adjusted_rand_score(y_true, labels)      # 1 = perfect, 0 = random

# 5. Choose k: loop over k values, plot inertia + silhouette
```

---
*QLSC 612 · Quantitative Life Sciences · McGill University*